In [ ]:
import pandas as pd
import numpy as np

In [ ]:
variant_df = pd.read_pickle("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/variant_info_annotated_extended_df.pkl")
print(variant_df.columns)
print(variant_df.head())


In [ ]:
homolog_df = pd.read_pickle("/mnt/d/phd/scripts/15_vertical_evolutionary_analysis_orthologs/data/output/combined_blast_results.pkl")
homolog_df['query_title'] = homolog_df['query_title'].str.replace(
    r'^(\w+)_(\d+)-(\d+)$',  # match UNIPROTID_START-END
    r'\1_\2_\3',              # replace with UNIPROTID_START_END
    regex=True
)
print(homolog_df.columns)
print(homolog_df.head())



In [ ]:
# extract protein id from homolog query_title (first part before first _)
homolog_proteins = set(homolog_df['query_title'].str.split('_').str[0].unique())

# get proteins from variant df
variant_proteins = set(variant_df['protein'].unique())

in_both = variant_proteins & homolog_proteins
only_in_variants = variant_proteins - homolog_proteins
only_in_homologs = homolog_proteins - variant_proteins

print(f"Proteins in both:       {len(in_both)}")
print(f"Only in variants:       {len(only_in_variants)}")
print(f"Only in homologs:       {len(only_in_homologs)}")

# spot check
print(f"\nSample variant proteins:  {list(variant_proteins)[:5]}")
print(f"Sample homolog proteins:  {list(homolog_proteins)[:5]}")

In [ ]:
# --- CHECK REGION ID OVERLAP BETWEEN DATASETS ---

region_ids_variants = set(variant_df['region_id'].unique())
region_ids_homologs = set(homolog_df['query_title'].unique())

in_both = region_ids_variants & region_ids_homologs
only_in_variants = region_ids_variants - region_ids_homologs
only_in_homologs = region_ids_homologs - region_ids_variants

print(f"Regions in both datasets:       {len(in_both)}")
print(f"Only in variant df:             {len(only_in_variants)}")
print(f"Only in homolog df:             {len(only_in_homologs)}")
print(f"\nTotal unique in variant df:     {len(region_ids_variants)}")
print(f"Total unique in homolog df:     {len(region_ids_homologs)}")

# --- SPOT CHECK a few from each exclusive set ---
if only_in_variants:
    print(f"\nSample only in variants:  {list(only_in_variants)[:5]}")
if only_in_homologs:
    print(f"Sample only in homologs:  {list(only_in_homologs)[:5]}")

In [ ]:
# ============================================================
# BUILD BASE FEATURE MATRIX WITH REGION IDS AND GROUP LABELS
# ============================================================

# get all unique regions and their group labels from both sources
regions_variants = (
    variant_df[['region_id', 'group']]
    .drop_duplicates()
    .rename(columns={'region_id': 'region_id', 'group': 'group_variants'})
)

regions_homologs = (
    homolog_df[['query_title', 'group']]
    .drop_duplicates()
    .rename(columns={'query_title': 'region_id', 'group': 'group_homologs'})
)

# outer merge to get all regions from both sources
regions_all = pd.merge(regions_variants, regions_homologs, on='region_id', how='outer')

# --- CHECK FOR CONFLICTS ---
conflicts = regions_all[
    regions_all['group_variants'].notna() & 
    regions_all['group_homologs'].notna() & 
    (regions_all['group_variants'] != regions_all['group_homologs'])
]
if len(conflicts) > 0:
    print(f"WARNING: {len(conflicts)} regions have conflicting group labels:")
    print(conflicts)
else:
    print("No group label conflicts found.")

# --- RESOLVE GROUP LABEL ---
# take whichever is available, variants takes priority if both present and agree
regions_all['group'] = regions_all['group_variants'].combine_first(regions_all['group_homologs'])

# final check - should be zero
print(f"\nRemaining NaN in group: {regions_all['group'].isna().sum()}")
print(f"Total regions: {len(regions_all)}")
print(f"\nGroup distribution:\n{regions_all['group'].value_counts()}")

# --- SET AS BASE FEATURE MATRIX ---
feature_matrix = regions_all[['region_id', 'group']].set_index('region_id')
print(f"\nBase feature matrix shape: {feature_matrix.shape}")

In [ ]:
# ============================================================
# ADD n_homologs AND n_variants TO FEATURE MATRIX
# ============================================================

# --- n_homologs from homolog df ---
n_homologs = (
    homolog_df
    .groupby('query_title')
    .size()
    .reset_index(name='n_homologs')
    .rename(columns={'query_title': 'region_id'})
    .set_index('region_id')
)

# --- n_variants from variant df ---
n_variants = (
    variant_df
    .groupby('region_id')
    .size()
    .reset_index(name='n_variants')
    .set_index('region_id')
)

# --- JOIN ONTO FEATURE MATRIX ---
feature_matrix = feature_matrix.join(n_homologs, how='left')
feature_matrix = feature_matrix.join(n_variants, how='left')

# --- LOG TRANSFORM ---
feature_matrix['log_n_homologs'] = np.log1p(feature_matrix['n_homologs'])
feature_matrix['log_n_variants'] = np.log1p(feature_matrix['n_variants'])

# --- SANITY CHECK ---
print(feature_matrix.shape)
print(f"\nMissing values:\n{feature_matrix.isna().sum()}")
print(f"\nFeature summary by group:")
print(feature_matrix.groupby('group')[['log_n_homologs', 'log_n_variants']].describe())

In [ ]:
# ============================================================
# ADD SPECIES BREADTH AND VARIANT DENSITY
# ============================================================

# --- species breadth from homolog df ---
species_breadth = (
    homolog_df
    .groupby('query_title')['species']
    .nunique()
    .reset_index(name='species_breadth')
    .rename(columns={'query_title': 'region_id'})
    .set_index('region_id')
)

# --- region length and variant density from variant df ---
region_length = (
    variant_df
    .groupby('region_id')
    .agg(region_length=('prot_region_start', lambda x: 
        variant_df.loc[x.index, 'prot_region_end'].iloc[0] - x.iloc[0]))
    .reset_index()
    .set_index('region_id')
)

# --- JOIN ---
feature_matrix = feature_matrix.join(species_breadth, how='left')
feature_matrix = feature_matrix.join(region_length, how='left')

# --- COMPUTE VARIANT DENSITY ---
feature_matrix['variant_density'] = (
    feature_matrix['n_variants'] / feature_matrix['region_length']
)

# --- SANITY CHECK ---
print(f"Missing values:\n{feature_matrix.isna().sum()}")
print(f"\nFeature summary by group:")
print(feature_matrix.groupby('group')[['species_breadth', 'variant_density']].describe())

In [ ]:
# ============================================================
# COLUMN REGISTRY
# ============================================================

# auxiliary columns - kept for reference, sanity checks, and interpretability
# never fed into PCA/random forest
aux_cols = [
    'n_homologs',
    'n_variants', 
    'region_length',
]

# feature columns - these go into the model
# update this list every time you add a new feature
feature_cols = [
    'log_n_homologs',
    'log_n_variants',
    'species_breadth',
    'variant_density',
]

# group label - used as y in the model, never as a feature
# 'group'

# quick check
print("Current feature matrix columns:")
print(f"  aux:      {aux_cols}")
print(f"  features: {feature_cols}")
print(f"  missing:  {feature_matrix[feature_cols].isna().sum().to_dict()}")

In [ ]:
# ============================================================
# RG DENSITY HELPER FUNCTION
# ============================================================
import ast
# drop if already exists (in case of re-running the cell)
cols_to_drop = ['frac_variants_at_rg', 'delta_rg_homologs']
feature_matrix = feature_matrix.drop(columns=[c for c in cols_to_drop if c in feature_matrix.columns])

# also reset feature_cols to avoid duplicates
feature_cols = [c for c in feature_cols if c not in cols_to_drop]

def rg_density(seq):
    """Count RG duplets (R immediately followed by G) normalized by sequence length."""
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    count = sum(1 for i in range(len(seq)-1) if seq[i] == 'R' and seq[i+1] == 'G')
    return count / len(seq)

def parse_rg(val):
    """Parse rg column whether it's already a list or a string representation."""
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        return ast.literal_eval(val)
    return []

# ============================================================
# FRAC VARIANTS AT RG (from variant_df)
# ============================================================

variant_df['rg_before_parsed'] = variant_df['rg_before'].apply(parse_rg)
variant_df['rg_after_parsed'] = variant_df['rg_after'].apply(parse_rg)
variant_df['affects_rg'] = (
    variant_df['rg_before_parsed'].apply(len) != variant_df['rg_after_parsed'].apply(len)
)

frac_variants_at_rg = (
    variant_df
    .groupby('region_id')['affects_rg']
    .mean()
    .reset_index(name='frac_variants_at_rg')
    .set_index('region_id')
)

# ============================================================
# DELTA RG HOMOLOGS (from homolog_df)
# ============================================================

homolog_df['rg_density_query'] = homolog_df['qseq'].apply(rg_density)
homolog_df['rg_density_homolog'] = homolog_df['hseq'].apply(rg_density)
homolog_df['delta_rg_homolog'] = (
    homolog_df['rg_density_homolog'] - homolog_df['rg_density_query']
)

delta_rg_homologs = (
    homolog_df
    .groupby('query_title')['delta_rg_homolog']
    .mean()
    .reset_index(name='delta_rg_homologs')
    .rename(columns={'query_title': 'region_id'})
    .set_index('region_id')
)

# ============================================================
# JOIN ONTO FEATURE MATRIX
# ============================================================

feature_matrix = feature_matrix.join(frac_variants_at_rg, how='left')
feature_matrix = feature_matrix.join(delta_rg_homologs, how='left')

feature_cols += ['frac_variants_at_rg', 'delta_rg_homologs']

# ============================================================
# SANITY CHECK
# ============================================================

print(f"Missing values:\n{feature_matrix[['frac_variants_at_rg', 'delta_rg_homologs']].isna().sum()}")
print(f"\nFeature summary by group:")
print(feature_matrix.groupby('group')[['frac_variants_at_rg', 'delta_rg_homologs']].describe())

In [ ]:
import pandas as pd

# ============================================================
# FULL SIGNIFICANT SUBSTITUTION LISTS
# ============================================================

# all 7 from variant analysis
all_sig_variants_subs_df = pd.DataFrame([
    ('M', 'V',  1.376613, 4.179648e-04),
    ('R', 'M', -1.331753, 3.301218e-07),
    ('N', 'K', -1.278503, 6.625371e-05),
    ('G', 'W', -1.175019, 2.013183e-13),
    ('G', 'S',  0.517264, 3.301218e-07),
    ('R', 'S', -0.516406, 2.528081e-05),
    ('R', 'Q',  0.465843, 1.396725e-05),
], columns=['from', 'to', 'log2_or', 'p_value'])
all_sig_variants_subs_df['abs_log2_or'] = all_sig_variants_subs_df['log2_or'].abs()

# all 43 from homolog analysis
all_sig_homologs_subs_df = pd.DataFrame([
    ('K', 'R', -7.378813, 5.480888e-45),
    ('I', 'V', -7.217554, 8.537247e-15),
    ('K', 'E',  7.129283, 1.200415e-49),
    ('I', 'T',  6.964135, 7.457358e-14),
    ('T', 'M', -6.375885, 4.109776e-32),
    ('E', 'Q', -5.984400, 1.485727e-15),
    ('E', 'G', -5.562662, 9.235533e-07),
    ('Y', 'H', -5.507795, 6.519207e-07),
    ('F', 'L', -4.502235, 2.362192e-14),
    ('G', 'R', -4.460177, 1.934023e-39),
    ('E', 'D',  4.426751, 4.559585e-27),
    ('F', 'V',  4.419100, 1.131790e-16),
    ('T', 'A',  4.169998, 6.635199e-52),
    ('T', 'V', -3.985673, 2.046209e-06),
    ('N', 'K', -3.952079, 2.649375e-04),
    ('L', 'M',  3.786743, 7.298480e-09),
    ('L', 'P', -3.756364, 1.194898e-10),
    ('S', 'L', -3.721405, 6.881132e-06),
    ('D', 'N', -3.624680, 5.355618e-15),
    ('V', 'M',  3.496884, 1.425002e-18),
    ('S', 'P', -3.447794, 3.641033e-22),
    ('G', 'E', -3.385263, 4.348674e-06),
    ('D', 'E',  3.293895, 9.916002e-20),
    ('A', 'S',  3.293040, 2.268309e-32),
    ('E', 'K', -3.188883, 6.193777e-07),
    ('S', 'N',  2.943367, 8.813962e-21),
    ('V', 'L', -2.938224, 1.313374e-04),
    ('T', 'I', -2.567972, 3.135503e-06),
    ('R', 'S',  2.547627, 1.020383e-09),
    ('V', 'A', -2.425306, 2.318886e-04),
    ('G', 'S',  2.414524, 6.974625e-30),
    ('R', 'G', -2.318138, 6.721622e-09),
    ('A', 'P', -2.264653, 2.115679e-05),
    ('V', 'G', -2.177341, 2.917501e-04),
    ('R', 'P', -2.083648, 1.270370e-07),
    ('R', 'H',  1.904942, 1.280560e-05),
    ('S', 'T',  1.840702, 5.228858e-04),
    ('R', 'K',  1.829844, 1.826510e-08),
    ('G', 'V',  1.566417, 2.989004e-05),
    ('Q', 'H',  1.534154, 2.138086e-05),
    ('P', 'S',  1.509091, 3.582501e-06),
    ('A', 'V', -1.097933, 1.559029e-05),
    ('A', 'G', -1.092757, 5.429865e-06),
], columns=['from', 'to', 'log2_or', 'p_value'])
all_sig_homologs_subs_df['abs_log2_or'] = all_sig_homologs_subs_df['log2_or'].abs()

# ============================================================
# DYNAMIC SUBSTITUTION SELECTION PER MODE
# ============================================================
def get_subs_for_mode(mode, variants_df, homologs_df, topn=7):
    def filter_df(df, mode):
        if mode == 'all':
            return df
        elif mode == 'top10':
            return df.nlargest(10, 'abs_log2_or')
        elif mode == 'top5':
            return df.nlargest(5, 'abs_log2_or')
        elif mode == 'top20':
            return df.nlargest(20, 'abs_log2_or')
        elif mode == 'top7':
            return df.nlargest(topn, 'abs_log2_or')
        elif mode == 'R_only':
            return df[(df['from'] == 'R') | (df['to'] == 'R')]
        elif mode == 'G_only':
            return df[(df['from'] == 'G') | (df['to'] == 'G')]
        elif mode == 'RG':
            return df[(df['from'].isin(['R','G'])) | (df['to'].isin(['R','G']))]
        elif mode == 'none':
            return df.iloc[0:0]  # empty
        else:
            raise ValueError(f"Unknown mode: {mode}")

    v = filter_df(variants_df, mode)
    h = filter_df(homologs_df, mode)
    return (
        list(zip(v['from'], v['to'])),
        list(zip(h['from'], h['to']))
    )

# --- test ---
for mode in ['top5', 'top7', 'top10', 'top20', 'all', 'R_only', 'G_only', 'RG', 'none']:
    v, h = get_subs_for_mode(mode, all_sig_variants_subs_df, all_sig_homologs_subs_df)
    print(f"{mode:8s} — variants: {len(v):2d}  homologs: {len(h):2d}")

In [ ]:
SUBSTITUTION_MODE = 'top7'

####### TEST


# ============================================================
# FULL PIPELINE - SINGLE RERUNNABLE BLOCK
# ============================================================
# Change SUBSTITUTION_MODE to experiment:
#   'top7'    — top 7 from each source (original)
#   'all'     — all significant substitutions
#   'R_only'  — substitutions involving R
#   'G_only'  — substitutions involving G
#   'RG'      — substitutions involving R or G
#   'none'    — no substitution features

# SUBSTITUTION_MODE = 'top7'

# ============================================================
# 0. IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from umap import UMAP

# ============================================================
# 1. SUBSTITUTION SELECTION
# ============================================================
# all_sig_variants_subs = [
#     ('M', 'V'), ('R', 'M'), ('N', 'K'), ('G', 'W'),
#     ('G', 'S'), ('R', 'S'), ('R', 'Q')
# ]
# all_sig_homolog_subs = [
#     ('K', 'R'), ('I', 'V'), ('K', 'E'), ('I', 'T'),
#     ('T', 'M'), ('E', 'Q'), ('E', 'G')
# ]

# sig_variants_subs, sig_homolog_subs = get_subs_for_mode(
#     SUBSTITUTION_MODE,
#     all_sig_variants_subs_df,
#     all_sig_homologs_subs_df
# )

# def filter_subs(subs, mode):
#     if mode in ('all', 'top7'):
#         return subs
#     elif mode == 'R_only':
#         return [(f, t) for f, t in subs if f == 'R' or t == 'R']
#     elif mode == 'G_only':
#         return [(f, t) for f, t in subs if f == 'G' or t == 'G']
#     elif mode == 'RG':
#         return [(f, t) for f, t in subs if f in ('R', 'G') or t in ('R', 'G')]
#     elif mode == 'none':
#         return []
#     else:
#         raise ValueError(f"Unknown mode: {mode}")

# sig_variants_subs = filter_subs(all_sig_variants_subs, SUBSTITUTION_MODE)
# sig_homolog_subs  = filter_subs(all_sig_homolog_subs,  SUBSTITUTION_MODE)

sig_variants_subs, sig_homolog_subs = get_subs_for_mode(
    SUBSTITUTION_MODE,
    all_sig_variants_subs_df,
    all_sig_homologs_subs_df
)

print(f"Mode: {SUBSTITUTION_MODE}")
print(f"Variant substitutions ({len(sig_variants_subs)}): {sig_variants_subs}")
print(f"Homolog substitutions ({len(sig_homolog_subs)}):  {sig_homolog_subs}")

# ============================================================
# 2. BUILD FEATURE COLS FROM SCRATCH
# ============================================================
base_feature_cols = [
    'log_n_homologs',
    'log_variant_density',
    'log_species_breadth',
    'frac_variants_at_rg',
    'delta_rg_homologs',
]

for col in ['variant_density', 'species_breadth']:
    feature_matrix[f'log_{col}'] = np.log1p(feature_matrix[col])
    # replace in feature_cols
    feature_cols = [f'log_{col}' if c == col else c for c in feature_cols]

# drop old substitution columns from feature_matrix
sub_cols_to_drop = [c for c in feature_matrix.columns if 'var_frac' in c or 'hom_frac' in c]
fm = feature_matrix.drop(columns=sub_cols_to_drop).copy()

def get_variant_aa_change(row):
    """Extract AA change for single nucleotide substitutions only."""
    # skip indels
    if len(str(row['REF'])) != 1 or len(str(row['ALT'])) != 1:
        return None
    if not isinstance(row['before_seq'], str) or not isinstance(row['after_seq'], str):
        return None
    prot_pos = int(row['mut_pos'] // 3)
    if prot_pos >= len(row['before_seq']) or prot_pos >= len(row['after_seq']):
        return None
    aa_from = row['before_seq'][prot_pos]
    aa_to = row['after_seq'][prot_pos]
    if aa_from == aa_to:
        return None  # synonymous
    return (aa_from, aa_to)

# compute and join variant substitution features
if 'aa_change_tuple' not in variant_df.columns:
    variant_df['aa_change_tuple'] = variant_df.apply(get_variant_aa_change, axis=1)

for aa_from, aa_to in sig_variants_subs:
    col = f'var_frac_{aa_from}_to_{aa_to}'
    variant_df[col] = variant_df['aa_change_tuple'] == (aa_from, aa_to)
    feat = (
        variant_df.groupby('region_id')[col]
        .mean()
        .reset_index(name=col)
        .set_index('region_id')
    )
    fm = fm.join(feat, how='left')

def get_homolog_aa_changes(qseq, hseq):
    """Extract all pairwise AA changes from aligned sequences, skipping gaps."""
    if not isinstance(qseq, str) or not isinstance(hseq, str):
        return []
    changes = []
    for q, h in zip(qseq, hseq):
        if q == '-' or h == '-':
            continue
        if q != h:
            changes.append((q, h))
    return changes

# compute and join homolog substitution features
if 'aa_changes' not in homolog_df.columns:
    homolog_df['aa_changes'] = homolog_df.apply(
        lambda row: get_homolog_aa_changes(row['qseq'], row['hseq']), axis=1
    )

for aa_from, aa_to in sig_homolog_subs:
    col = f'hom_frac_{aa_from}_to_{aa_to}'
    homolog_df[col] = homolog_df['aa_changes'].apply(
        lambda changes: changes.count((aa_from, aa_to)) / len(changes) if len(changes) > 0 else 0
    )
    feat = (
        homolog_df.groupby('query_title')[col]
        .mean()
        .reset_index(name=col)
        .rename(columns={'query_title': 'region_id'})
        .set_index('region_id')
    )
    fm = fm.join(feat, how='left')

sub_feature_cols = [f'var_frac_{f}_to_{t}' for f, t in sig_variants_subs] + \
                   [f'hom_frac_{f}_to_{t}' for f, t in sig_homolog_subs]
feature_cols_run = base_feature_cols + sub_feature_cols

print(f"\nTotal features: {len(feature_cols_run)}")

# ============================================================
# 3. CLEANUP
# ============================================================
# cap outliers
for col in feature_cols_run:
    cap = fm[col].quantile(0.99)
    fm[col] = fm[col].clip(upper=cap)

# median imputation
for col in feature_cols_run:
    n_missing = fm[col].isna().sum()
    if n_missing > 0:
        fm[col] = fm[col].fillna(fm[col].median())

# scale
X = fm[feature_cols_run].copy()
y = (fm['group'] == 'positive').astype(int)

scaler = StandardScaler()
X_scaled_run = pd.DataFrame(
    scaler.fit_transform(X),
    index=X.index,
    columns=X.columns
)

# ============================================================
# 4. PCA
# ============================================================
pca = PCA()
X_pca = pca.fit_transform(X_scaled_run)
cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'PCA — substitution mode: {SUBSTITUTION_MODE}', fontsize=13)

ax = axes[0]
ax.bar(range(1, len(pca.explained_variance_ratio_)+1),
       pca.explained_variance_ratio_, color='steelblue', alpha=0.7)
ax.plot(range(1, len(cumvar)+1), cumvar, 'o-', color='darkorange')
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('PC')
ax.set_ylabel('Variance Explained')
ax.set_title('Scree Plot')

ax = axes[1]
colors = y.map({1: '#2ca02c', 0: '#d62728'})
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6, s=30, edgecolors='none')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PC1 vs PC2')
ax.legend(handles=[mpatches.Patch(color='#2ca02c', label='positive'),
                   mpatches.Patch(color='#d62728', label='negative')])

ax = axes[2]
loadings = pd.DataFrame(pca.components_[:2].T,
                        index=feature_cols_run,
                        columns=['PC1', 'PC2']).sort_values('PC1')
ax.barh(loadings.index, loadings['PC1'], color='steelblue', alpha=0.7, label='PC1')
ax.barh(loadings.index, loadings['PC2'], color='darkorange', alpha=0.5, label='PC2')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('Loadings PC1 & PC2')
ax.legend()

plt.tight_layout()
plt.show()

# ============================================================
# 5. RANDOM FOREST
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=200, max_features='sqrt', random_state=42)

oof_probs = np.zeros(len(y))
fold_importances = []
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled_run, y)):
    X_train, X_val = X_scaled_run.iloc[train_idx], X_scaled_run.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    rf.fit(X_train, y_train)
    oof_probs[val_idx] = rf.predict_proba(X_val)[:, 1]
    fold_aucs.append(roc_auc_score(y_val, oof_probs[val_idx]))
    fold_importances.append(rf.feature_importances_)

overall_auc = roc_auc_score(y, oof_probs)
print(f"\nMode: {SUBSTITUTION_MODE}")
print(f"OOF ROC-AUC: {overall_auc:.3f}")
print(f"Per-fold:    {[f'{a:.3f}' for a in fold_aucs]}")
print(f"Mean ± std:  {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")

# feature importances
importances = pd.DataFrame(fold_importances, columns=feature_cols_run)
imp_mean = importances.mean().sort_values(ascending=False)
imp_std  = importances.std()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Random Forest — substitution mode: {SUBSTITUTION_MODE}', fontsize=13)

ax = axes[0]
ax.barh(imp_mean.index, imp_mean.values,
        xerr=imp_std[imp_mean.index].values,
        color='steelblue', alpha=0.7, capsize=3)
ax.invert_yaxis()
ax.set_xlabel('Mean Importance')
ax.set_title('Feature Importances')

ax = axes[1]
RocCurveDisplay.from_predictions(y, oof_probs, ax=ax, color='steelblue')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_title(f'ROC Curve (AUC = {overall_auc:.3f})')

plt.tight_layout()
plt.show()

# ============================================================
# 5b. SHAP ANALYSIS
# ============================================================
import shap

# Refit RF on full dataset for SHAP (cross-val already gave us OOF performance)
rf_full = RandomForestClassifier(n_estimators=200, max_features='sqrt', random_state=42)
rf_full.fit(X_scaled_run, y)

explainer = shap.TreeExplainer(rf_full)
shap_values = explainer(X_scaled_run)  # returns Explanation object

# For binary RF, shap_values has shape (n_samples, n_features, 2)
# We take class 1 (positive) slice
shap_vals_pos = shap_values[..., 1]
shap_df = pd.DataFrame(shap_vals_pos.values, columns=feature_cols_run, index=X_scaled_run.index)
# ── Plot 1: Beeswarm summary ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
shap.plots.beeswarm(shap_vals_pos, max_display=len(feature_cols_run), show=False)
plt.title(f'SHAP Beeswarm — substitution mode: {SUBSTITUTION_MODE}')
plt.tight_layout()
plt.show()

# ── Plot 2: Bar summary (mean |SHAP|) ───────────────────────
n = len(shap_df)
shap_means = shap_df.abs().mean().sort_values(ascending=False)
shap_stds  = shap_df.abs().std()[shap_means.index]
shap_ci95  = 1.96 * shap_stds / np.sqrt(n)

fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(shap_means.index[::-1], shap_means.values[::-1],
        xerr=shap_ci95.values[::-1],
        color='steelblue', alpha=0.7, capsize=3, ecolor='gray', linewidth=0.8)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title(f'SHAP Feature Importance ± 95% CI ({SUBSTITUTION_MODE})')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

# # ── Plot 3: SHAP dependence plots for top 4 features ────────

# top_features = shap_df.abs().mean().sort_values(ascending=False).head(4).index.tolist()

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle(f'SHAP Dependence Plots — top 4 features ({SUBSTITUTION_MODE})', fontsize=13)

# for ax, feat in zip(axes.flatten(), top_features):
#     shap.plots.scatter(
#         shap_vals_pos[:, feature_cols_run.index(feat)],
#         color=shap_vals_pos,
#         ax=ax,
#         show=False
#     )
#     ax.set_title(feat)

# plt.tight_layout()
# plt.show()

# ── Plot 4: SHAP interaction heatmap ────────────────────────
# # Computationally heavier — comment out if slow
# shap_interaction_values = explainer.shap_interaction_values(X_scaled_run)
# # For binary RF, returns a list of 2 arrays (one per class), each (n_samples, n_features, n_features)
# # Take class 1 (positive)
# interaction_mean = np.abs(shap_interaction_values[1]).mean(axis=0)
# interaction_df = pd.DataFrame(interaction_mean, index=feature_cols_run, columns=feature_cols_run)

# fig, ax = plt.subplots(figsize=(12, 10))
# im = ax.imshow(interaction_df.values, cmap='viridis', aspect='auto')
# ax.set_xticks(range(len(feature_cols_run)))
# ax.set_yticks(range(len(feature_cols_run)))
# ax.set_xticklabels(feature_cols_run, rotation=90, fontsize=8)
# ax.set_yticklabels(feature_cols_run, fontsize=8)
# plt.colorbar(im, ax=ax, label='Mean |SHAP interaction|')
# ax.set_title(f'SHAP Interaction Heatmap — substitution mode: {SUBSTITUTION_MODE}')
# plt.tight_layout()
# plt.show()

# # ── Plot 5: Per-sample SHAP for FN candidates ───────────────
# # Find top 10 FN candidates (negatives with highest OOF score)
# # We use oof_probs from section 5 here (before the refit)
# fn_top_idx = (
#     fm[fm['group'] == 'negative']
#     .assign(fn_score_tmp=oof_probs[fm['group'].values == 'negative'])
#     .nlargest(10, 'fn_score_tmp')
#     .index
# )

# fn_shap = shap_vals_pos[X_scaled_run.index.isin(fn_top_idx)]

# fig, ax = plt.subplots(figsize=(10, 6))
# shap.plots.bar(fn_shap, max_display=len(feature_cols_run), show=False)
# plt.title(f'SHAP — Top 10 FN candidates ({SUBSTITUTION_MODE})')
# plt.tight_layout()
# plt.show()

# ── Store SHAP values for downstream use ────────────────────
fm['shap_max_feature'] = shap_df.abs().idxmax(axis=1)
fm['shap_max_value']   = shap_df.abs().max(axis=1)

print(f"\nSHAP analysis complete. Top features by mean |SHAP|:")
print(shap_df.abs().mean().sort_values(ascending=False).round(4).to_string())

# ── Plot 5b: Per-candidate SHAP heatmap for top FN ──────────
fn_mask = fm['group'] == 'negative'
fn_indices = fm[fn_mask].index
fn_oof = oof_probs[fn_mask.values]

top10_fn_index = pd.Series(fn_oof, index=fn_indices).nlargest(10).index

fn_shap_df = shap_df.loc[top10_fn_index].copy()
fn_shap_df.index = [f"{idx}\n(score={oof_probs[X_scaled_run.index.get_loc(idx)]:.2f})" 
                     for idx in top10_fn_index]

fig, ax = plt.subplots(figsize=(14, 6))
vmax = fn_shap_df.abs().values.max()
im = ax.imshow(fn_shap_df.values, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(feature_cols_run)))
ax.set_yticks(range(len(top10_fn_index)))
ax.set_xticklabels(feature_cols_run, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(fn_shap_df.index, fontsize=8)
plt.colorbar(im, ax=ax, label='SHAP value (→ positive)')
ax.set_title(f'Per-FN-candidate SHAP — top 10 ({SUBSTITUTION_MODE})')
plt.tight_layout()
plt.show()

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

# ============================================================
# 6. FN / FP CANDIDATES + UMAP
# ============================================================
fm['fn_score'] = oof_probs

# FN candidates
fn_candidates = fm[fm['group'] == 'negative'].sort_values('fn_score', ascending=False)
fp_candidates = fm[fm['group'] == 'positive'].sort_values('fn_score', ascending=True)

print(f"\nTop 10 FN candidates (mode: {SUBSTITUTION_MODE}):")
print(fn_candidates[['fn_score'] + feature_cols_run].head(10).round(3).to_string())

print(f"\nTop 10 FP candidates (mode: {SUBSTITUTION_MODE}):")
print(fp_candidates[['fn_score'] + feature_cols_run].head(10).round(3).to_string())

# UMAP
umap_model = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_umap = umap_model.fit_transform(X_scaled_run)
umap_df = pd.DataFrame(X_umap, index=X_scaled_run.index, columns=['UMAP1', 'UMAP2'])

def classify(row):
    if row['group'] == 'positive' and row['fn_score'] >= 0.25:
        return 'true_positive'
    elif row['group'] == 'positive' and row['fn_score'] < 0.25:
        return 'false_positive'
    elif row['group'] == 'negative' and row['fn_score'] >= 0.75:
        return 'false_negative'
    else:
        return 'true_negative'

fm['classification'] = fm.apply(classify, axis=1)
print(f"\nClassification counts:\n{fm['classification'].value_counts()}")


#### UMAP

from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 8))

# Plot positives and negatives
for group, color, label in [('positive', '#2ca02c', 'positive'), ('negative', '#d62728', 'negative')]:
    mask = fm['group'] == group
    subset = umap_df.loc[fm[mask].index]
    ax.scatter(subset['UMAP1'], subset['UMAP2'],
               c=color, alpha=0.4, s=25, edgecolors='none', label=label, zorder=2)

# ── Manual offsets: adjust these to move labels ─────────────
# Format: 'region_id': (delta_x, delta_y)
label_offsets = {idx: (-6.0, 0) for i, idx in enumerate(top10_fn_index)}
# ────────────────────────────────────────────────────────────

# label_offsets = {
#     'Q5J8M3_19_39':      (-6.0, 2.0),
#     'Q5J8M3_19_39':      (-6.0, 2.0),
#     # etc.
# }

label_offsets["Q5J8M3_19_39"] = (-6.0, 2.0)
label_offsets["Q00536_0_40"] = (-6.0, 1.2)
label_offsets["Q2Y0W8_53_97"] = (-6.0, 1.8)
label_offsets["Q9UKV0_107_139"] = (-6.0, 1.4)
label_offsets["Q86UK7_310_352"] = (-6.0, 1.0)
label_offsets["P22735_57_92"] = (-6.0, 0.4)
label_offsets["Q86SG6_275_305"] = (-6.0, -0.4)

for idx in top10_fn_index:
    x, y = umap_df.loc[idx, 'UMAP1'], umap_df.loc[idx, 'UMAP2']
    score = oof_probs[X_scaled_run.index.get_loc(idx)]
    dx, dy = label_offsets[idx]
    tx, ty = x + dx, y + dy

    # Dot with border
    ax.scatter(x, y, c='#d62728', s=80, edgecolors='black', linewidths=1.4, zorder=4)

    # Connecting line
    ax.annotate(
        "",
        xy=(x, y), xytext=(tx, ty),
        arrowprops=dict(arrowstyle='-', color='black', lw=0.8),
        zorder=3
    )

    # Label box
    ax.text(
        tx, ty,
        f"{idx} ({score:.2f})",
        fontsize=10,
        ha='right',
        va='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', lw=0.6, alpha=0.9),
        zorder=5
    )

ax.set_title(f'UMAP — top 10 FN candidates highlighted ({SUBSTITUTION_MODE})')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend()
plt.tight_layout()
plt.show()

# fig, ax = plt.subplots(figsize=(9, 7))
# for label, color in color_map.items():
#     mask = fm['classification'] == label
#     subset = umap_df.loc[fm[mask].index]
#     ax.scatter(subset['UMAP1'], subset['UMAP2'],
#                c=color, alpha=0.6, s=30, edgecolors='none', label=label)

# ax.set_title(f'UMAP — substitution mode: {SUBSTITUTION_MODE}')
# ax.set_xlabel('UMAP 1')
# ax.set_ylabel('UMAP 2')
# ax.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
import plotly.graph_objects as go

# ── Build hover info ─────────────────────────────────────────
hover_df = umap_df.copy()
hover_df['group']      = fm['group']
hover_df['fn_score']   = oof_probs
hover_df['region_id']  = fm.index
hover_df['is_top_fn']  = hover_df.index.isin(top10_fn_index)

# Add feature values for hover
for col in feature_cols_run:
    hover_df[col] = fm[col]

def make_hover(row):
    lines = [
        f"<b>{row['region_id']}</b>",
        f"Group: {row['group']}",
        f"FN score: {row['fn_score']:.3f}",
        "─────────────────",
    ] + [f"{col}: {row[col]:.3f}" for col in feature_cols_run]
    return "<br>".join(lines)

hover_df['hover_text'] = hover_df.apply(make_hover, axis=1)

# ── Build figure ─────────────────────────────────────────────
fig = go.Figure()

# Positives
pos_mask = hover_df['group'] == 'positive'
fig.add_trace(go.Scatter(
    x=hover_df.loc[pos_mask, 'UMAP1'],
    y=hover_df.loc[pos_mask, 'UMAP2'],
    mode='markers',
    name='positive',
    marker=dict(color='#2ca02c', size=6, opacity=0.5,
                line=dict(width=0)),
    text=hover_df.loc[pos_mask, 'hover_text'],
    hoverinfo='text'
))

# Negatives (non-FN)
neg_mask = (hover_df['group'] == 'negative') & (~hover_df['is_top_fn'])
fig.add_trace(go.Scatter(
    x=hover_df.loc[neg_mask, 'UMAP1'],
    y=hover_df.loc[neg_mask, 'UMAP2'],
    mode='markers',
    name='negative',
    marker=dict(color='#d62728', size=6, opacity=0.5,
                line=dict(width=0)),
    text=hover_df.loc[neg_mask, 'hover_text'],
    hoverinfo='text'
))

# Top 10 FN candidates
fn_mask = hover_df['is_top_fn']
fig.add_trace(go.Scatter(
    x=hover_df.loc[fn_mask, 'UMAP1'],
    y=hover_df.loc[fn_mask, 'UMAP2'],
    mode='markers+text',
    name='top 10 FN',
    marker=dict(color='#d62728', size=10, opacity=1.0,
                line=dict(color='black', width=1.5)),
    text=hover_df.loc[fn_mask, 'region_id'],
    textposition='top center',
    textfont=dict(size=8),
    hovertext=hover_df.loc[fn_mask, 'hover_text'],
    hoverinfo='text'
))

fig.update_layout(
    title=f'UMAP — top 10 FN candidates highlighted ({SUBSTITUTION_MODE})',
    xaxis_title='UMAP 1',
    yaxis_title='UMAP 2',
    width=900, height=700,
    hoverlabel=dict(bgcolor='white', font_size=11, font_family='monospace'),
    legend=dict(itemsizing='constant')
)

fig.show()

In [ ]:
# ============================================================
# FULL PIPELINE - XGBOOST
# ============================================================
# Change SUBSTITUTION_MODE to experiment:
#   'top7'    — top 7 from each source
#   'all'     — all significant substitutions
#   'R_only'  — substitutions involving R
#   'G_only'  — substitutions involving G
#   'RG'      — substitutions involving R or G
#   'none'    — no substitution features

SUBSTITUTION_MODE = 'RG'

# ============================================================
# 0. IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from umap import UMAP
from xgboost import XGBClassifier

# ============================================================
# 1. SUBSTITUTION SELECTION
# ============================================================
sig_variants_subs, sig_homolog_subs = get_subs_for_mode(
    SUBSTITUTION_MODE,
    all_sig_variants_subs_df,
    all_sig_homologs_subs_df
)

print(f"Mode: {SUBSTITUTION_MODE}")
print(f"Variant substitutions ({len(sig_variants_subs)}): {sig_variants_subs}")
print(f"Homolog substitutions ({len(sig_homolog_subs)}):  {sig_homolog_subs}")

# ============================================================
# 2. BUILD FEATURE COLS FROM SCRATCH
# ============================================================
base_feature_cols = [
    'log_n_homologs',
    'log_variant_density',
    'log_species_breadth',
    'frac_variants_at_rg',
    'delta_rg_homologs',
]

sub_cols_to_drop = [c for c in feature_matrix.columns if 'var_frac' in c or 'hom_frac' in c]
fm = feature_matrix.drop(columns=sub_cols_to_drop).copy()

if 'aa_change_tuple' not in variant_df.columns:
    variant_df['aa_change_tuple'] = variant_df.apply(get_variant_aa_change, axis=1)

for aa_from, aa_to in sig_variants_subs:
    col = f'var_frac_{aa_from}_to_{aa_to}'
    variant_df[col] = variant_df['aa_change_tuple'] == (aa_from, aa_to)
    feat = (
        variant_df.groupby('region_id')[col]
        .mean()
        .reset_index(name=col)
        .set_index('region_id')
    )
    fm = fm.join(feat, how='left')

if 'aa_changes' not in homolog_df.columns:
    homolog_df['aa_changes'] = homolog_df.apply(
        lambda row: get_homolog_aa_changes(row['qseq'], row['hseq']), axis=1
    )

for aa_from, aa_to in sig_homolog_subs:
    col = f'hom_frac_{aa_from}_to_{aa_to}'
    homolog_df[col] = homolog_df['aa_changes'].apply(
        lambda changes: changes.count((aa_from, aa_to)) / len(changes) if len(changes) > 0 else 0
    )
    feat = (
        homolog_df.groupby('query_title')[col]
        .mean()
        .reset_index(name=col)
        .rename(columns={'query_title': 'region_id'})
        .set_index('region_id')
    )
    fm = fm.join(feat, how='left')

sub_feature_cols = [f'var_frac_{f}_to_{t}' for f, t in sig_variants_subs] + \
                   [f'hom_frac_{f}_to_{t}' for f, t in sig_homolog_subs]
feature_cols_run = base_feature_cols + sub_feature_cols

print(f"\nTotal features: {len(feature_cols_run)}")

# ============================================================
# 3. CLEANUP
# ============================================================
for col in feature_cols_run:
    cap = fm[col].quantile(0.99)
    fm[col] = fm[col].clip(upper=cap)

for col in feature_cols_run:
    if fm[col].isna().sum() > 0:
        fm[col] = fm[col].fillna(fm[col].median())

X = fm[feature_cols_run].copy()
y = (fm['group'] == 'positive').astype(int)

scaler = StandardScaler()
X_scaled_run = pd.DataFrame(
    scaler.fit_transform(X),
    index=X.index,
    columns=X.columns
)

# ============================================================
# 4. PCA
# ============================================================
pca = PCA()
X_pca = pca.fit_transform(X_scaled_run)
cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'PCA — substitution mode: {SUBSTITUTION_MODE}', fontsize=13)

ax = axes[0]
ax.bar(range(1, len(pca.explained_variance_ratio_)+1),
       pca.explained_variance_ratio_, color='steelblue', alpha=0.7)
ax.plot(range(1, len(cumvar)+1), cumvar, 'o-', color='darkorange')
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('PC')
ax.set_ylabel('Variance Explained')
ax.set_title('Scree Plot')

ax = axes[1]
colors = y.map({1: '#2ca02c', 0: '#d62728'})
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6, s=30, edgecolors='none')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PC1 vs PC2')
ax.legend(handles=[mpatches.Patch(color='#2ca02c', label='positive'),
                   mpatches.Patch(color='#d62728', label='negative')])

ax = axes[2]
loadings = pd.DataFrame(pca.components_[:2].T,
                        index=feature_cols_run,
                        columns=['PC1', 'PC2']).sort_values('PC1')
ax.barh(loadings.index, loadings['PC1'], color='steelblue', alpha=0.7, label='PC1')
ax.barh(loadings.index, loadings['PC2'], color='darkorange', alpha=0.5, label='PC2')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('Loadings PC1 & PC2')
ax.legend()

plt.tight_layout()
plt.show()

# ============================================================
# 5. XGBOOST
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# scale_pos_weight balances pos/neg class sizes
neg_count = (y == 0).sum()
pos_count = (y == 1).sum()

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=neg_count / pos_count,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

oof_probs = np.zeros(len(y))
fold_importances = []
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled_run, y)):
    X_train, X_val = X_scaled_run.iloc[train_idx], X_scaled_run.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    xgb.fit(X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False)
    oof_probs[val_idx] = xgb.predict_proba(X_val)[:, 1]
    fold_aucs.append(roc_auc_score(y_val, oof_probs[val_idx]))
    fold_importances.append(xgb.feature_importances_)

overall_auc = roc_auc_score(y, oof_probs)
print(f"\nXGBoost — Mode: {SUBSTITUTION_MODE}")
print(f"OOF ROC-AUC: {overall_auc:.3f}")
print(f"Per-fold:    {[f'{a:.3f}' for a in fold_aucs]}")
print(f"Mean ± std:  {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")

# feature importances
importances = pd.DataFrame(fold_importances, columns=feature_cols_run)
imp_mean = importances.mean().sort_values(ascending=False)
imp_std  = importances.std()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'XGBoost — substitution mode: {SUBSTITUTION_MODE}', fontsize=13)

ax = axes[0]
ax.barh(imp_mean.index, imp_mean.values,
        xerr=imp_std[imp_mean.index].values,
        color='darkorange', alpha=0.7, capsize=3)
ax.invert_yaxis()
ax.set_xlabel('Mean Importance')
ax.set_title('Feature Importances')

ax = axes[1]
RocCurveDisplay.from_predictions(y, oof_probs, ax=ax, color='darkorange')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_title(f'ROC Curve (AUC = {overall_auc:.3f})')

plt.tight_layout()
plt.show()

# ============================================================
# 6. FN / FP CANDIDATES + UMAP
# ============================================================
fm['fn_score'] = oof_probs

fn_candidates = fm[fm['group'] == 'negative'].sort_values('fn_score', ascending=False)
fp_candidates = fm[fm['group'] == 'positive'].sort_values('fn_score', ascending=True)

print(f"\nTop 10 FN candidates (XGBoost — mode: {SUBSTITUTION_MODE}):")
print(fn_candidates[['fn_score'] + base_feature_cols].head(10).round(3).to_string())

print(f"\nTop 10 FP candidates (XGBoost — mode: {SUBSTITUTION_MODE}):")
print(fp_candidates[['fn_score'] + base_feature_cols].head(10).round(3).to_string())

# UMAP
umap_model = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_umap = umap_model.fit_transform(X_scaled_run)
umap_df = pd.DataFrame(X_umap, index=X_scaled_run.index, columns=['UMAP1', 'UMAP2'])

def classify(row):
    if row['group'] == 'positive' and row['fn_score'] >= 0.5:
        return 'true_positive'
    elif row['group'] == 'positive' and row['fn_score'] < 0.5:
        return 'false_positive'
    elif row['group'] == 'negative' and row['fn_score'] >= 0.5:
        return 'false_negative'
    else:
        return 'true_negative'

fm['classification'] = fm.apply(classify, axis=1)
print(f"\nClassification counts:\n{fm['classification'].value_counts()}")

color_map = {
    'true_positive':  '#2ca02c',
    'true_negative':  '#d62728',
    'false_negative': '#ff7f0e',
    'false_positive': '#9467bd',
}

fig, ax = plt.subplots(figsize=(9, 7))
for label, color in color_map.items():
    mask = fm['classification'] == label
    subset = umap_df.loc[fm[mask].index]
    ax.scatter(subset['UMAP1'], subset['UMAP2'],
               c=color, alpha=0.6, s=30, edgecolors='none', label=label)

ax.set_title(f'UMAP — XGBoost — substitution mode: {SUBSTITUTION_MODE}')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
umap_df = pd.DataFrame(X_umap, index=X_scaled.index, columns=['UMAP1', 'UMAP2'])

In [ ]:
# ============================================================
# FALSE POSITIVE CANDIDATES
# ============================================================

# get positive regions sorted by fn_score ascending
fp_candidates = feature_matrix[feature_matrix['group'] == 'positive'].copy()
fp_candidates = fp_candidates.sort_values('fn_score', ascending=True)

# add per-feature percentile relative to negative group
neg_features = feature_matrix[feature_matrix['group'] == 'negative'][feature_cols]

for col in feature_cols:
    fp_candidates[f'{col}_pct_vs_neg'] = fp_candidates[col].apply(
        lambda x: (neg_features[col] < x).mean()
    )

print(f"Top 20 FP candidates:\n")
print(fp_candidates[['fn_score'] + feature_cols].head(20).to_string())

# ============================================================
# UPDATED UMAP - ALL FOUR CATEGORIES
# ============================================================

# classify each region
def classify(row):
    if row['group'] == 'positive' and row['fn_score'] >= 0.5:
        return 'true_positive'
    elif row['group'] == 'positive' and row['fn_score'] < 0.5:
        return 'false_positive'
    elif row['group'] == 'negative' and row['fn_score'] >= 0.5:
        return 'false_negative'
    else:
        return 'true_negative'

feature_matrix['classification'] = feature_matrix.apply(classify, axis=1)

print(f"\nClassification counts:")
print(feature_matrix['classification'].value_counts())

# plot
color_map = {
    'true_positive':  '#2ca02c',  # green
    'true_negative':  '#d62728',  # red
    'false_negative': '#ff7f0e',  # orange
    'false_positive': '#9467bd',  # purple
}

fig, ax = plt.subplots(figsize=(10, 8))
for label, color in color_map.items():
    mask = feature_matrix['classification'] == label
    idx = feature_matrix[mask].index
    umap_subset = umap_df.loc[idx]
    ax.scatter(umap_subset['UMAP1'], umap_subset['UMAP2'],
               c=color, alpha=0.6, s=30, edgecolors='none', label=label)

ax.set_title('UMAP: TP / TN / FN / FP candidates')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
##### BACKUP

In [ ]:
# ============================================================
# SUBSTITUTION FEATURES
# ============================================================

# --- SIGNIFICANT SUBSTITUTIONS TO TRACK ---
# top 7 from variants (by abs log2_or)
sig_variants_subs = [
    ('M', 'V'), ('R', 'M'), ('N', 'K'), ('G', 'W'),
    ('G', 'S'), ('R', 'S'), ('R', 'Q')
]

# top 7 from homologs (by abs log2_or)
sig_homolog_subs = [
    ('K', 'R'), ('I', 'V'), ('K', 'E'), ('I', 'T'),
    ('T', 'M'), ('E', 'Q'), ('E', 'G')
]

# ============================================================
# VARIANT SUBSTITUTIONS
# ============================================================

def get_variant_aa_change(row):
    """Extract AA change for single nucleotide substitutions only."""
    # skip indels
    if len(str(row['REF'])) != 1 or len(str(row['ALT'])) != 1:
        return None
    if not isinstance(row['before_seq'], str) or not isinstance(row['after_seq'], str):
        return None
    prot_pos = int(row['mut_pos'] // 3)
    if prot_pos >= len(row['before_seq']) or prot_pos >= len(row['after_seq']):
        return None
    aa_from = row['before_seq'][prot_pos]
    aa_to = row['after_seq'][prot_pos]
    if aa_from == aa_to:
        return None  # synonymous
    return (aa_from, aa_to)

variant_df['aa_change_tuple'] = variant_df.apply(get_variant_aa_change, axis=1)

# compute frac per significant substitution per region
for aa_from, aa_to in sig_variants_subs:
    col = f'var_frac_{aa_from}_to_{aa_to}'
    variant_df[col] = variant_df['aa_change_tuple'] == (aa_from, aa_to)
    feat = (
        variant_df
        .groupby('region_id')[col]
        .mean()
        .reset_index(name=col)
        .set_index('region_id')
    )
    # defensive drop
    if col in feature_matrix.columns:
        feature_matrix = feature_matrix.drop(columns=[col])
    feature_matrix = feature_matrix.join(feat, how='left')
    feature_cols.append(col)

# ============================================================
# HOMOLOG SUBSTITUTIONS
# ============================================================

def get_homolog_aa_changes(qseq, hseq):
    """Extract all pairwise AA changes from aligned sequences, skipping gaps."""
    if not isinstance(qseq, str) or not isinstance(hseq, str):
        return []
    changes = []
    for q, h in zip(qseq, hseq):
        if q == '-' or h == '-':
            continue
        if q != h:
            changes.append((q, h))
    return changes

homolog_df['aa_changes'] = homolog_df.apply(
    lambda row: get_homolog_aa_changes(row['qseq'], row['hseq']), axis=1
)

# compute frac per significant substitution per region
for aa_from, aa_to in sig_homolog_subs:
    col = f'hom_frac_{aa_from}_to_{aa_to}'
    homolog_df[col] = homolog_df['aa_changes'].apply(
        lambda changes: changes.count((aa_from, aa_to)) / len(changes) if len(changes) > 0 else 0
    )
    feat = (
        homolog_df
        .groupby('query_title')[col]
        .mean()
        .reset_index(name=col)
        .rename(columns={'query_title': 'region_id'})
        .set_index('region_id')
    )
    if col in feature_matrix.columns:
        feature_matrix = feature_matrix.drop(columns=[col])
    feature_matrix = feature_matrix.join(feat, how='left')
    feature_cols.append(col)

# ============================================================
# SANITY CHECK
# ============================================================

sub_cols = [c for c in feature_cols if 'frac' in c and 'to' in c]
print(f"Added {len(sub_cols)} substitution features")
print(f"\nMissing values:\n{feature_matrix[sub_cols].isna().sum()}")
print(f"\nFeature summary by group:")
print(feature_matrix.groupby('group')[sub_cols].mean().T)

# ============================================================
# CLEANUP
# ============================================================
import numpy as np
from sklearn.preprocessing import StandardScaler

# --- drop duplicate region_id column if present ---
if 'region_id' in feature_matrix.columns:
    feature_matrix = feature_matrix.drop(columns=['region_id'])

# --- confirm feature_cols is clean and matches matrix ---
feature_cols = [c for c in feature_cols if c in feature_matrix.columns]
print(f"Features going into model: {len(feature_cols)}")
print(feature_cols)

# --- cap outliers at 99th percentile per feature ---
for col in feature_cols:
    cap = feature_matrix[col].quantile(0.99)
    feature_matrix[col] = feature_matrix[col].clip(upper=cap)

# --- features that still need log transform ---
# variant_density and species_breadth are right-skewed, log transform them
for col in ['variant_density', 'species_breadth']:
    feature_matrix[f'log_{col}'] = np.log1p(feature_matrix[col])
    # replace in feature_cols
    feature_cols = [f'log_{col}' if c == col else c for c in feature_cols]

# --- median imputation per feature ---
imputation_report = {}
for col in feature_cols:
    n_missing = feature_matrix[col].isna().sum()
    if n_missing > 0:
        median_val = feature_matrix[col].median()
        feature_matrix[col] = feature_matrix[col].fillna(median_val)
        imputation_report[col] = {'n_imputed': n_missing, 'median_used': median_val}

print(f"\nImputed features:")
for col, info in imputation_report.items():
    print(f"  {col}: {info['n_imputed']} values imputed with median {info['median_used']:.4f}")

# --- StandardScaler normalization ---
# keep unscaled version for interpretability
X = feature_matrix[feature_cols].copy()
y = (feature_matrix['group'] == 'positive').astype(int)

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    index=X.index,
    columns=X.columns
)

# --- final check ---
print(f"\nX_scaled shape: {X_scaled.shape}")
print(f"y distribution:\n{y.value_counts()}")
print(f"\nAny remaining NaNs: {X_scaled.isna().sum().sum()}")


# ============================================================
# PCA SANITY CHECK
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA


feature_cols = [c for c in feature_cols if c != 'log_n_variants']
# --- fit PCA ---
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# --- variance explained ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# plot 1: scree plot
ax = axes[0]
cumvar = np.cumsum(pca.explained_variance_ratio_)
ax.bar(range(1, len(pca.explained_variance_ratio_)+1),
       pca.explained_variance_ratio_, color='steelblue', alpha=0.7, label='per component')
ax.plot(range(1, len(cumvar)+1), cumvar, 'o-', color='darkorange', label='cumulative')
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained')
ax.set_title('Scree Plot')
ax.legend()

# plot 2: PC1 vs PC2 colored by group
ax = axes[1]
colors = y.map({1: '#2ca02c', 0: '#d62728'})
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6, s=30, edgecolors='none')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PC1 vs PC2')
pos_patch = mpatches.Patch(color='#2ca02c', label='positive')
neg_patch = mpatches.Patch(color='#d62728', label='negative')
ax.legend(handles=[pos_patch, neg_patch])

# plot 3: feature loadings for PC1 and PC2
ax = axes[2]
loadings = pd.DataFrame(
    pca.components_[:2].T,
    index=feature_cols,
    columns=['PC1', 'PC2']
).sort_values('PC1')
ax.barh(loadings.index, loadings['PC1'], color='steelblue', alpha=0.7, label='PC1')
ax.barh(loadings.index, loadings['PC2'], color='darkorange', alpha=0.5, label='PC2')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Loading')
ax.set_title('Feature Loadings PC1 & PC2')
ax.legend()

plt.tight_layout()
plt.show()

# --- numerical summary ---
print("Variance explained per component:")
for i, var in enumerate(pca.explained_variance_ratio_[:5]):
    print(f"  PC{i+1}: {var*100:.1f}% (cumulative: {cumvar[i]*100:.1f}%)")

print(f"\nFeatures with highest PC1 loadings:")
print(loadings['PC1'].abs().sort_values(ascending=False).head(5))

# remove log_n_variants from feature_cols

feature_cols = [c for c in feature_cols if c != 'log_n_variants']
# rerun PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled[feature_cols])

cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# scree plot
ax = axes[0]
ax.bar(range(1, len(pca.explained_variance_ratio_)+1),
       pca.explained_variance_ratio_, color='steelblue', alpha=0.7, label='per component')
ax.plot(range(1, len(cumvar)+1), cumvar, 'o-', color='darkorange', label='cumulative')
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained')
ax.set_title('Scree Plot')
ax.legend()

# PC1 vs PC2
ax = axes[1]
colors = y.map({1: '#2ca02c', 0: '#d62728'})
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6, s=30, edgecolors='none')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PC1 vs PC2')
pos_patch = mpatches.Patch(color='#2ca02c', label='positive')
neg_patch = mpatches.Patch(color='#d62728', label='negative')
ax.legend(handles=[pos_patch, neg_patch])

# loadings
ax = axes[2]
loadings = pd.DataFrame(
    pca.components_[:2].T,
    index=feature_cols,
    columns=['PC1', 'PC2']
).sort_values('PC1')
ax.barh(loadings.index, loadings['PC1'], color='steelblue', alpha=0.7, label='PC1')
ax.barh(loadings.index, loadings['PC2'], color='darkorange', alpha=0.5, label='PC2')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Loading')
ax.set_title('Feature Loadings PC1 & PC2')
ax.legend()

plt.tight_layout()
plt.show()

print("Variance explained per component:")
for i, var in enumerate(pca.explained_variance_ratio_[:5]):
    print(f"  PC{i+1}: {var*100:.1f}% (cumulative: {cumvar[i]*100:.1f}%)")

print(f"\nFeatures with highest PC1 loadings:")
print(loadings['PC1'].abs().sort_values(ascending=False).head(5))

# ============================================================
# RANDOM FOREST WITH STRATIFIED 5-FOLD CV
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt

X_model = X_scaled[feature_cols]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=200, max_features='sqrt', random_state=42)

# collect out-of-fold predictions and feature importances
oof_probs = np.zeros(len(y))
fold_importances = []
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_model, y)):
    X_train, X_val = X_model.iloc[train_idx], X_model.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf.fit(X_train, y_train)
    oof_probs[val_idx] = rf.predict_proba(X_val)[:, 1]
    fold_aucs.append(roc_auc_score(y_val, oof_probs[val_idx]))
    fold_importances.append(rf.feature_importances_)

# overall OOF AUC
overall_auc = roc_auc_score(y, oof_probs)
print(f"OOF ROC-AUC: {overall_auc:.3f}")
print(f"Per-fold AUC: {[f'{a:.3f}' for a in fold_aucs]}")
print(f"Mean: {np.mean(fold_aucs):.3f} +/- {np.std(fold_aucs):.3f}")

# ============================================================
# FEATURE IMPORTANCES
# ============================================================
importances = pd.DataFrame(
    fold_importances,
    columns=feature_cols
)
imp_mean = importances.mean().sort_values(ascending=False)
imp_std = importances.std()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# feature importances with error bars
ax = axes[0]
ax.barh(imp_mean.index, imp_mean.values,
        xerr=imp_std[imp_mean.index].values,
        color='steelblue', alpha=0.7, capsize=3)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Mean Importance (across folds)')
ax.set_title('Feature Importances')
ax.invert_yaxis()

# ROC curve
ax = axes[1]
RocCurveDisplay.from_predictions(y, oof_probs, ax=ax, color='steelblue')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_title(f'ROC Curve (OOF AUC = {overall_auc:.3f})')

plt.tight_layout()
plt.show()

# ============================================================
# STORE OOF PROBABILITIES IN FEATURE MATRIX
# ============================================================
feature_matrix['fn_score'] = oof_probs
# ============================================================
# RANKED FN CANDIDATE LIST
# ============================================================

# get negative regions only
fn_candidates = feature_matrix[feature_matrix['group'] == 'negative'].copy()

# sort by fn_score descending
fn_candidates = fn_candidates.sort_values('fn_score', ascending=False)

# add per-feature percentile relative to positive group
pos_features = feature_matrix[feature_matrix['group'] == 'positive'][feature_cols]

for col in feature_cols:
    fn_candidates[f'{col}_pct'] = fn_candidates[col].apply(
        lambda x: (pos_features[col] < x).mean()
    )

# summary table: score + raw features + percentiles
pct_cols = [f'{col}_pct' for col in feature_cols]

print(f"Top 20 FN candidates:\n")
print(fn_candidates[['fn_score'] + feature_cols].head(20).to_string())

# ============================================================
# UMAP VISUALIZATION
# ============================================================
from umap import UMAP

umap = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_umap = umap.fit_transform(X_scaled[feature_cols])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# plot 1: colored by group label
ax = axes[0]
colors = y.map({1: '#2ca02c', 0: '#d62728'})
ax.scatter(X_umap[:, 0], X_umap[:, 1], c=colors, alpha=0.6, s=30, edgecolors='none')
pos_patch = mpatches.Patch(color='#2ca02c', label='positive')
neg_patch = mpatches.Patch(color='#d62728', label='negative')
ax.legend(handles=[pos_patch, neg_patch])
ax.set_title('UMAP colored by group')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')

# plot 2: colored by fn_score (neg only highlighted)
ax = axes[1]
# plot positives in green first
pos_mask = y == 1
ax.scatter(X_umap[pos_mask, 0], X_umap[pos_mask, 1],
           c='#2ca02c', alpha=0.3, s=20, edgecolors='none', label='positive')

# plot negatives colored by fn_score
neg_mask = y == 0
sc = ax.scatter(X_umap[neg_mask, 0], X_umap[neg_mask, 1],
                c=feature_matrix.loc[y[neg_mask].index, 'fn_score'],
                cmap='RdYlGn', alpha=0.8, s=40, edgecolors='none',
                vmin=0, vmax=1)
plt.colorbar(sc, ax=ax, label='FN score')
ax.set_title('UMAP: negative regions colored by FN score')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend()

plt.tight_layout()
plt.show()

# ============================================================
# TOP 10 FN CANDIDATES - CLEAN OUTPUT
# ============================================================

top10_fn = fn_candidates[['fn_score'] + feature_cols].head(10).copy()

# round for readability
top10_fn = top10_fn.round(3)

print("TOP 10 FALSE NEGATIVE CANDIDATES")
print("="*60)
for region_id, row in top10_fn.iterrows():
    protein = region_id.split('_')[0]
    start = region_id.split('_')[1]
    end = region_id.split('_')[2]
    print(f"\nRank {list(top10_fn.index).index(region_id)+1}: {region_id}")
    print(f"  Protein:        {protein}")
    print(f"  Region:         {start}-{end}")
    print(f"  FN score:       {row['fn_score']}")
    print(f"  log_n_homologs: {row['log_n_homologs']:.3f}  |  log_variant_density: {row['log_variant_density']:.3f}")
    print(f"  species_breadth:{row['log_species_breadth']:.3f}  |  delta_rg_homologs:   {row['delta_rg_homologs']:.3f}")
    print(f"  frac_at_rg:     {row['frac_variants_at_rg']:.3f}")